In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Lambda, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import skipgrams
from tensorflow.keras.utils import to_categorical


In [2]:
text = "the cat sits on the mat the cat eats fish"

# Tokenize the text
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

word_index = tokenizer.word_index
vocab_size = len(word_index) + 1

print("Vocabulary:", word_index)
print("Vocabulary Size:", vocab_size)


Vocabulary: {'the': 1, 'cat': 2, 'sits': 3, 'on': 4, 'mat': 5, 'eats': 6, 'fish': 7}
Vocabulary Size: 8


In [3]:
sequence = tokenizer.texts_to_sequences([text])[0]

# b. Generate training data
window_size = 2
contexts = []
targets = []

for i in range(window_size, len(sequence) - window_size):
    context = (
        sequence[i - window_size:i]
        + sequence[i + 1:i + window_size + 1]
    )
    target = sequence[i]

    contexts.append(context)
    targets.append(target)

X = np.array(contexts)
y = to_categorical(targets, num_classes=vocab_size)

print("\nTraining Contexts:", X.shape)
print("Training Targets:", y.shape)



Training Contexts: (6, 4)
Training Targets: (6, 8)


In [4]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=10),
    Lambda(lambda x: tf.reduce_mean(x, axis=1)),
    Dense(vocab_size, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

model.fit(X, y, epochs=200, verbose=0)

print("\nModel training completed!")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)



Model training completed!


In [5]:
predictions = model.predict(X, verbose=0)

for i in range(len(X)):
    context_words = [
        tokenizer.index_word.get(idx, "?") for idx in X[i]
    ]

    predicted_id = np.argmax(predictions[i])
    predicted_word = tokenizer.index_word.get(predicted_id, "?")
    actual_word = tokenizer.index_word.get(targets[i], "?")

    print("\nContext words:", context_words)
    print("Actual target word:", actual_word)
    print("Predicted target word:", predicted_word)


Context words: ['the', 'cat', 'on', 'the']
Actual target word: sits
Predicted target word: the

Context words: ['cat', 'sits', 'the', 'mat']
Actual target word: on
Predicted target word: the

Context words: ['sits', 'on', 'mat', 'the']
Actual target word: the
Predicted target word: the

Context words: ['on', 'the', 'the', 'cat']
Actual target word: mat
Predicted target word: the

Context words: ['the', 'mat', 'cat', 'eats']
Actual target word: the
Predicted target word: the

Context words: ['mat', 'the', 'eats', 'fish']
Actual target word: cat
Predicted target word: cat
